# BraTS 2024 Post-Treatment Glioma Segmentation\n## 2.5D U-Net | 5 Adjacent Slices | 5 Classes

In [ ]:
import os, glob, random, gc
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from scipy import ndimage
from scipy.ndimage import zoom
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
CONFIG = {
    # Data
    'data_root': '/kaggle/input/datasets/i212385nomanarif/2024-brats-glioma',
    'modalities': ['t1n', 't1c', 't2w', 't2f'],
    'seg_suffix': 'seg',
    
    # 2.5D settings
    'k_2p5d': 2,          # neighbor slices each side
    'n_slices': 5,        # 2*k + 1
    'in_channels': 20,    # 4 modalities x 5 slices
    
    # Image
    'slice_size': 240,
    
    # Classes: 0=BG, 1=ET, 2=NETC, 3=SNFH, 4=RC
    'num_classes': 5,
    'class_names': ['Background', 'ET', 'NETC', 'SNFH', 'RC'],
    
    # Model
    'base_channels': 32,
    
    # Training
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'batch_size': 8,      # adjust to GPU
    'num_epochs': 200,
    'val_split': 0.2,
    'seed': 42,
    
    # Loss weights
    'dice_weight': 0.5,
    'ce_weight': 0.5,
    
    # Saving
    'checkpoint_dir': './checkpoints',
    'best_model_path': './checkpoints/best_unet2p5d.pth',
}

os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
print('Config loaded.')
print(f"In channels: {CONFIG['in_channels']}  |  Out classes: {CONFIG['num_classes']}")


In [ ]:
def get_patient_list(data_root):
    """Scan data_root for valid patient folders."""
    patients = []
    for folder in sorted(glob.glob(os.path.join(data_root, '*'))):
        if not os.path.isdir(folder):
            continue
        pid = os.path.basename(folder)
        files = {}
        for mod in CONFIG['modalities']:
            pattern = os.path.join(folder, f'{pid}*{mod}*.nii.gz')
            found = glob.glob(pattern)
            if found:
                files[mod] = found[0]
        seg_pattern = os.path.join(folder, f'{pid}*{CONFIG["seg_suffix"]}*.nii.gz')
        seg_found = glob.glob(seg_pattern)
        if len(files) == 4 and seg_found:
            files['seg'] = seg_found[0]
            patients.append({'id': pid, 'files': files, 'folder': folder})
    return patients

def load_brats_volume(patient_info):
    volumes = []
    for mod in CONFIG['modalities']:
        path = patient_info['files'][mod]
        vol = nib.load(path).get_fdata().astype(np.float32)
        volumes.append(vol)
    volume = np.stack(volumes, axis=-1)
    mask = nib.load(patient_info['files']['seg']).get_fdata().astype(np.int64)
    return volume, mask

def normalize_slice_wise(volume):
    D, C, H, W = volume.shape
    for z in range(D):
        for c in range(C):
            slice_ = volume[z, c]
            brain_mask = slice_ > 0
            if np.any(brain_mask):
                mean = slice_[brain_mask].mean()
                std = slice_[brain_mask].std()
                if std > 0:
                    volume[z, c] = np.where(brain_mask, (slice_ - mean) / std, 0)
    return volume

def crop_brain_roi(volume, mask, margin=5, min_brain_ratio=0.01):
    brain_intensity = np.max(volume, axis=1)  # (D, H, W)
    mean = np.mean(brain_intensity)
    std  = np.std(brain_intensity)
    thresh = mean + 0.5 * std
    brain_mask = brain_intensity > thresh

    labeled, num = ndimage.label(brain_mask)
    if num > 0:
        sizes = ndimage.sum(brain_mask, labeled, range(1, num + 1))
        largest_cc = (sizes.argmax() + 1)
        brain_mask = (labeled == largest_cc)

    slice_ratio = np.mean(brain_mask, axis=(1, 2))
    valid_slices = slice_ratio > min_brain_ratio

    if not np.any(valid_slices):
        return volume, mask

    z_idx = np.where(valid_slices)[0]
    zmin, zmax = z_idx.min(), z_idx.max()

    brain_mask_valid = brain_mask[zmin:zmax+1]
    coords = np.where(brain_mask_valid)
    ymin, ymax = coords[1].min(), coords[1].max()
    xmin, xmax = coords[2].min(), coords[2].max()

    zmin = max(zmin - margin, 0)
    ymin = max(ymin - margin, 0)
    xmin = max(xmin - margin, 0)

    zmax = min(zmax + margin, volume.shape[0] - 1)
    ymax = min(ymax + margin, volume.shape[2] - 1)
    xmax = min(xmax + margin, volume.shape[3] - 1)

    volume = volume[zmin:zmax+1, :, ymin:ymax+1, xmin:xmax+1]
    mask   = mask[zmin:zmax+1, ymin:ymax+1, xmin:xmax+1]

    return volume, mask

def resize_volume(volume, mask, target_size=(CONFIG['slice_size'], CONFIG['slice_size'])):
    D, C, H, W = volume.shape
    scale_h = target_size[0] / H
    scale_w = target_size[1] / W

    volume_resized = np.zeros((D, C, target_size[0], target_size[1]), dtype=volume.dtype)
    for d in range(D):
        for c in range(C):
            volume_resized[d, c] = zoom(volume[d, c], (scale_h, scale_w), order=1)

    mask_resized = np.zeros((D, target_size[0], target_size[1]), dtype=mask.dtype)
    for d in range(D):
        mask_resized[d] = zoom(mask[d], (scale_h, scale_w), order=0)

    return volume_resized, mask_resized

def create_2p5d_slices(volume, mask, k_2p5d=CONFIG['k_2p5d']):
    D, C, H, W = volume.shape
    slices = []
    masks = []
    
    for z in range(k_2p5d, D - k_2p5d):
        slice_stack = []
        for offset in range(-k_2p5d, k_2p5d + 1):
            slice_stack.append(volume[z + offset])  # (C, H, W)
        
        slice_2p5d = np.concatenate(slice_stack, axis=0) # (C*(2k+1), H, W)
        slices.append(slice_2p5d)
        masks.append(mask[z])

    return np.array(slices), np.array(masks)

print('Data utilities (ROI crop, Resize, Norm, 2.5D) defined.')


In [ ]:
class BraTS2024Dataset2p5D(Dataset):
    def __init__(self, slices, masks, augment=False):
        self.slices = slices
        self.masks = masks
        self.augment = augment
        
    def __len__(self):
        return len(self.slices)
    
    def __getitem__(self, idx):
        x = self.slices[idx].astype(np.float32)
        y = self.masks[idx].astype(np.int64)
        
        if self.augment:
            if random.random() > 0.5:
                x = x[:, :, ::-1].copy()
                y = y[:, ::-1].copy()
            if random.random() > 0.5:
                x = x[:, ::-1, :].copy()
                y = y[::-1, :].copy()
                
        return torch.from_numpy(x), torch.from_numpy(y)

def prepare_2p5d_data(patient_list, desc="Processing"):
    all_slices = []
    all_masks = []
    
    for pinfo in tqdm(patient_list, desc=desc):
        try:
            # Load
            volume, mask = load_brats_volume(pinfo)
            # volume: (H, W, D, C) -> (D, C, H, W)
            volume = np.transpose(volume, (2, 3, 0, 1))
            mask = np.transpose(mask, (2, 0, 1))
            
            # ROI Crop
            volume, mask = crop_brain_roi(volume, mask)
            # Resize
            volume, mask = resize_volume(volume, mask)
            # Normalize Slice-wise
            volume = normalize_slice_wise(volume)
            # 2.5D Slices
            slices, slice_masks = create_2p5d_slices(volume, mask)
            
            if len(slices) == 0:
                continue
                
            all_slices.append(slices)
            all_masks.append(slice_masks)
            
            del volume, mask, slices, slice_masks
            gc.collect()
            
        except Exception as e:
            print(f"Error on {pinfo['id']}: {e}")
            continue
            
    if len(all_slices) == 0:
        return np.array([]), np.array([])
        
    return np.concatenate(all_slices, axis=0), np.concatenate(all_masks, axis=0)

def build_dataloaders(cfg):
    patients = get_patient_list(cfg['data_root'])
    print(f'Found {len(patients)} valid patients.')
    if len(patients) == 0:
        print("Please check your dataset path!")
        return None, None, None, None
        
    random.shuffle(patients)
    n_val = int(len(patients) * cfg['val_split'])
    val_patients = patients[:n_val]
    train_patients = patients[n_val:]
    
    print("Preparing Training Data...")
    tr_slices, tr_masks = prepare_2p5d_data(train_patients, "Train")
    
    print("Preparing Validation Data...")
    vl_slices, vl_masks = prepare_2p5d_data(val_patients, "Val")
    
    train_ds = BraTS2024Dataset2p5D(tr_slices, tr_masks, augment=True)
    val_ds   = BraTS2024Dataset2p5D(vl_slices, vl_masks, augment=False)
    
    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg['batch_size'], shuffle=False, num_workers=4, pin_memory=True)
    
    return train_loader, val_loader, train_patients, val_patients

print('Dataset loader defined.')


In [ ]:
# ─────────────────────────────────────────────
# 2.5D U-Net  (pure, no deep supervision)
# in_channels = 20  |  out_channels = 5
# ─────────────────────────────────────────────

class ConvBNReLU(nn.Module):
    def __init__(self, in_c, out_c, k=3, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, k, padding=p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, k, padding=p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class EncoderBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = ConvBNReLU(in_c, out_c)
        self.pool = nn.MaxPool2d(2, 2)
    def forward(self, x):
        skip = self.conv(x)
        down = self.pool(skip)
        return skip, down

class DecoderBlock(nn.Module):
    def __init__(self, in_c, skip_c, out_c):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_c, in_c // 2, 2, stride=2)
        self.conv = ConvBNReLU(in_c // 2 + skip_c, out_c)
    def forward(self, x, skip):
        x = self.up(x)
        dy = skip.size(2) - x.size(2)
        dx = skip.size(3) - x.size(3)
        x = F.pad(x, [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2])
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)

class UNet2p5D(nn.Module):
    """
    2.5D U-Net model.
    in_channels  = 20  (4 modalities x 5 slices)
    out_channels = 5   (BG, ET, NETC, SNFH, RC)
    base_channels = 32
    """
    def __init__(self, in_channels=20, out_channels=5, base=32):
        super().__init__()
        b = base
        # Encoder
        self.enc1 = EncoderBlock(in_channels, b)      
        self.enc2 = EncoderBlock(b,     b*2)           
        self.enc3 = EncoderBlock(b*2,   b*4)           
        self.enc4 = EncoderBlock(b*4,   b*8)           
        # Bottleneck
        self.bottleneck = ConvBNReLU(b*8, b*16)        
        # Decoder
        self.dec4 = DecoderBlock(b*16, b*8,  b*8)      
        self.dec3 = DecoderBlock(b*8,  b*4,  b*4)      
        self.dec2 = DecoderBlock(b*4,  b*2,  b*2)      
        self.dec1 = DecoderBlock(b*2,  b,    b)        
        # Output
        self.head = nn.Conv2d(b, out_channels, 1)
    
    def forward(self, x):
        s1, x = self.enc1(x)
        s2, x = self.enc2(x)
        s3, x = self.enc3(x)
        s4, x = self.enc4(x)
        x = self.bottleneck(x)
        x = self.dec4(x, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)
        return self.head(x)   # [B, 5, H, W]

# Quick sanity check
_m = UNet2p5D(in_channels=CONFIG['in_channels'],
              out_channels=CONFIG['num_classes'],
              base=CONFIG['base_channels'])
_x = torch.randn(2, 20, 240, 240)
_y = _m(_x)
total_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Model: UNet2p5D')
print(f'Output shape : {_y.shape}')
print(f'Parameters   : {total_params / 1e6:.2f} M')
del _m, _x, _y


In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, num_classes, smooth=1e-5, ignore_bg=True):
        super().__init__()
        self.C = num_classes
        self.smooth = smooth
        self.ignore_bg = ignore_bg
    
    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        one_hot = F.one_hot(targets, self.C).permute(0, 3, 1, 2).float()
        
        start_c = 1 if self.ignore_bg else 0
        dice_per_class = []
        for c in range(start_c, self.C):
            p = probs[:, c]
            g = one_hot[:, c]
            inter = (p * g).sum()
            union = p.sum() + g.sum()
            dice = (2 * inter + self.smooth) / (union + self.smooth)
            dice_per_class.append(1.0 - dice)
        return torch.stack(dice_per_class).mean()

class CombinedLoss(nn.Module):
    def __init__(self, num_classes, dice_w=0.5, ce_w=0.5):
        super().__init__()
        self.dice = DiceLoss(num_classes)
        self.ce   = nn.CrossEntropyLoss()
        self.dice_w = dice_w
        self.ce_w   = ce_w
    
    def forward(self, logits, targets):
        d = self.dice(logits, targets)
        c = self.ce(logits, targets)
        return self.dice_w * d + self.ce_w * c, d.item(), c.item()

print('Loss functions defined.')


In [ ]:
def dice_per_class(preds, targets, num_classes, smooth=1e-5):
    scores = []
    for c in range(1, num_classes):
        p = (preds == c).astype(float)
        g = (targets == c).astype(float)
        inter = (p * g).sum()
        union = p.sum() + g.sum()
        scores.append((2 * inter + smooth) / (union + smooth))
    return scores

def hd95_per_class(preds, targets, num_classes):
    try:
        from scipy.spatial.distance import directed_hausdorff
    except ImportError:
        return [float('nan')] * (num_classes - 1)
    
    scores = []
    for c in range(1, num_classes):
        p_pts = np.argwhere(preds == c)
        g_pts = np.argwhere(targets == c)
        if len(p_pts) == 0 and len(g_pts) == 0:
            scores.append(0.0)
        elif len(p_pts) == 0 or len(g_pts) == 0:
            scores.append(float('nan'))
        else:
            d1 = directed_hausdorff(p_pts, g_pts)[0]
            d2 = directed_hausdorff(g_pts, p_pts)[0]
            scores.append(max(d1, d2))
    return scores

print('Metrics defined.')


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = total_dice = total_ce = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss, d, c = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        total_dice += d
        total_ce   += c
    n = len(loader)
    return total_loss / n, total_dice / n, total_ce / n

@torch.no_grad()
def validate(model, loader, criterion, device, cfg):
    model.eval()
    total_loss = 0.0
    all_dice = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss, _, _ = criterion(logits, y)
        total_loss += loss.item()
        preds = logits.argmax(dim=1).cpu().numpy()
        targets = y.cpu().numpy()
        for b in range(preds.shape[0]):
            all_dice.append(dice_per_class(preds[b], targets[b], cfg['num_classes']))
    mean_dice = np.nanmean(all_dice, axis=0)
    return total_loss / len(loader), mean_dice

def run_training(cfg):
    train_loader, val_loader, train_pts, val_pts = build_dataloaders(cfg)
    if train_loader is None: return None, None
    
    model = UNet2p5D(cfg['in_channels'], cfg['num_classes'], cfg['base_channels']).to(device)
    optimizer = AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg['num_epochs'], eta_min=1e-6)
    criterion = CombinedLoss(cfg['num_classes'], cfg['dice_weight'], cfg['ce_weight'])
    
    best_val_dice = -1.0
    history = {'train_loss': [], 'val_loss': [], 'val_dice': []}
    
    for epoch in range(1, cfg['num_epochs'] + 1):
        tr_loss, tr_dice, tr_ce = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_dice = validate(model, val_loader, criterion, device, cfg)
        scheduler.step()
        
        mean_val_dice = val_dice.mean()
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['val_dice'].append(mean_val_dice)
        
        if mean_val_dice > best_val_dice:
            best_val_dice = mean_val_dice
            torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                        'val_dice': best_val_dice}, cfg['best_model_path'])
        
        class_names = cfg['class_names'][1:]
        dice_str = ' | '.join([f'{n}: {d:.4f}' for n, d in zip(class_names, val_dice)])
        print(f'[E{epoch:03d}/{cfg["num_epochs"]}] '
              f'TrLoss={tr_loss:.4f} VlLoss={val_loss:.4f} | '
              f'{dice_str} | Mean={mean_val_dice:.4f}'
              + (' ★' if mean_val_dice >= best_val_dice else ''))
    
    return model, history

# model, history = run_training(CONFIG)


## Inference and 3D Evaluation

In [ ]:
from scipy.ndimage import label
import scipy.ndimage as ndimage

def keep_largest_connected_component(mask):
    out_mask = np.zeros_like(mask)
    for c in range(1, 5):
        class_mask = mask == c
        if not np.any(class_mask): continue
        labeled, num_features = label(class_mask)
        if num_features == 0: continue
        largest_cc = labeled == (np.bincount(labeled.flat)[1:].argmax() + 1)
        out_mask[largest_cc] = c
    return out_mask

@torch.no_grad()
def predict_patient_volume(model, patient_info, cfg, device):
    """
    Predict the entire 3D volume for a single patient by processing 2.5D slices.
    Tích hợp bộ lọc làm mịn 3D Gaussian trên trục Z để khắc phục hiện tượng răng cưa (Lego Effect).
    """
    model.eval()
    k = cfg['k_2p5d']
    
    # 1. Load and preprocess volume
    volume, mask = load_brats_volume(patient_info)
    volume = np.transpose(volume, (2, 3, 0, 1)) # (D, C, H, W)
    mask = np.transpose(mask, (2, 0, 1))
    
    original_shape = mask.shape
    
    volume, mask = crop_brain_roi(volume, mask)
    volume, mask = resize_volume(volume, mask)
    volume = normalize_slice_wise(volume)
    
    # 2. Create 2.5D slices
    slices, slice_masks = create_2p5d_slices(volume, mask, k)
    if len(slices) == 0:
        return None, None
        
    slices = torch.tensor(slices, dtype=torch.float32) # Keep on CPU to avoid OOM
    
    # 3. Batch prediction
    probs = []
    batch_size = cfg['batch_size'] * 2 # Can use larger batch size for inference
    
    for i in range(0, len(slices), batch_size):
        batch = slices[i:i+batch_size].to(device)
        logits = model(batch)
        # Chuyển đổi logits thành xác suất (probabilities) để áp dụng bộ lọc mượt mà
        prob = torch.softmax(logits, dim=1).cpu().numpy()
        probs.append(prob)
        
    probs = np.concatenate(probs, axis=0) # (N, C, H, W)
    
    # [CẢI TIẾN] Bộ lọc làm mịn trục Z thích ứng (Class-Aware Z-axis Gaussian Smoothing)
    # Tránh làm mất các vùng u nhỏ (ET, NETC) bằng cách dùng sigma nhỏ, và làm mịn mạnh cho vùng to (SNFH)
    sigmas = {
        0: 1.0,  # Background
        1: 0.8,  # NETC (Lõi hoại tử - nhỏ) -> Làm mịn nhẹ
        2: 1.5,  # SNFH (Phù nề - to) -> Làm mịn mạnh để bo viền mượt mà
        3: 0.8,  # ET (U hoạt hóa - nhỏ) -> Làm mịn nhẹ
        4: 1.0   # RC (Hốc mổ) -> Làm mịn vừa
    }
    
    smoothed_probs = np.zeros_like(probs)
    for c in range(probs.shape[1]):
        sz = sigmas.get(c, 1.0)
        # sigma=(sz, 0.2, 0.2) nghĩa là làm mịn sz dọc trục Z, và cực kỳ nhẹ (0.2) trên mặt phẳng XY để giữ nguyên chi tiết
        smoothed_probs[:, c] = ndimage.gaussian_filter(probs[:, c], sigma=(sz, 0.2, 0.2))
        
    preds = smoothed_probs.argmax(axis=1) # (N, H, W)
    
    del slices, volume, mask, probs, smoothed_probs
    import gc; gc.collect()
    torch.cuda.empty_cache()
    return preds, slice_masks

def calculate_merged_dice(pred, target, classes_to_merge):
    p_mask = np.isin(pred, classes_to_merge)
    t_mask = np.isin(target, classes_to_merge)
    inter = np.logical_and(p_mask, t_mask).sum()
    union = p_mask.sum() + t_mask.sum()
    if union == 0: 
        return 1.0 # CHUẨN BRATS: Nếu GT không có u và AI cũng không vẽ -> Đoán đúng 100% -> Dice 1.0
    return 2.0 * inter / union

def calculate_merged_hd95(pred, target, classes_to_merge):
    try:
        from scipy.spatial.distance import directed_hausdorff
    except ImportError:
        return float('nan')
    
    p_pts = np.argwhere(np.isin(pred, classes_to_merge))
    g_pts = np.argwhere(np.isin(target, classes_to_merge))
    
    if len(p_pts) == 0 and len(g_pts) == 0:
        return 0.0 # CHUẨN BRATS: Nếu rỗng cả 2 -> Sai số = 0
    elif len(p_pts) == 0 or len(g_pts) == 0:
        return 374.0 # CHUẨN BRATS: Nếu 1 bên có u 1 bên không -> Phạt tối đa (Đường chéo não)
    else:
        d1 = directed_hausdorff(p_pts, g_pts)[0]
        d2 = directed_hausdorff(g_pts, p_pts)[0]
        return max(d1, d2)

def evaluate_test_set(model, test_patients, cfg, device):
    all_dice, all_hd95, all_iou, all_sens, all_prec = [], [], [], [], []
    all_wt_dice, all_tc_dice, all_et_dice = [], [], []
    all_wt_hd95, all_tc_hd95, all_et_hd95 = [], [], []
    
    print(f"Evaluating 3D Volumes for {len(test_patients)} patients in TEST SET...")
    from tqdm import tqdm
    for pinfo in tqdm(test_patients, desc="Test Patients"):
        pred_vol, gt_vol = predict_patient_volume(model, pinfo, cfg, device)
        if pred_vol is None: continue
            
        pred_vol = remove_small_connected_components(pred_vol) # Bộ lọc rác theo loại u
        
        # 1. Các chỉ số class độc lập (Cơ bản)
        dice = dice_per_class(pred_vol, gt_vol, cfg['num_classes'])
        hd95 = hd95_per_class(pred_vol, gt_vol, cfg['num_classes'])
        iou = iou_per_class(pred_vol, gt_vol, cfg['num_classes'])
        sens = sensitivity_per_class(pred_vol, gt_vol, cfg['num_classes'])
        prec = precision_per_class(pred_vol, gt_vol, cfg['num_classes'])
        
        # 2. CHUẨN QUỐC TẾ BRATS: Các vùng gộp (Merged Regions)
        # DICE
        wt_dice = calculate_merged_dice(pred_vol, gt_vol, [1, 2, 3])
        tc_dice = calculate_merged_dice(pred_vol, gt_vol, [1, 3])
        et_dice = calculate_merged_dice(pred_vol, gt_vol, [3])
        # HD95
        wt_hd95 = calculate_merged_hd95(pred_vol, gt_vol, [1, 2, 3])
        tc_hd95 = calculate_merged_hd95(pred_vol, gt_vol, [1, 3])
        et_hd95 = calculate_merged_hd95(pred_vol, gt_vol, [3])
        
        all_dice.append(dice); all_hd95.append(hd95); all_iou.append(iou); all_sens.append(sens); all_prec.append(prec)
        all_wt_dice.append(wt_dice); all_tc_dice.append(tc_dice); all_et_dice.append(et_dice)
        all_wt_hd95.append(wt_hd95); all_tc_hd95.append(tc_hd95); all_et_hd95.append(et_hd95)
        
    mean_dice, mean_hd95 = np.nanmean(all_dice, axis=0), np.nanmean(all_hd95, axis=0)
    mean_iou = np.nanmean(all_iou, axis=0)
    mean_sens, mean_prec = np.nanmean(all_sens, axis=0), np.nanmean(all_prec, axis=0)
    
    print("\n" + "="*80)
    print("🏆 BẢNG KẾT QUẢ ĐÁNH GIÁ 3D FULL VOLUME (CLASS ĐỘC LẬP)")
    print("="*80)
    for i, name in enumerate(cfg['class_names'][1:]):
        print(f"Class: {name:4s} | Dice: {mean_dice[i]:.4f} | HD95: {mean_hd95[i]:7.2f} | IoU: {mean_iou[i]:.4f} | Sens: {mean_sens[i]:.4f} | Prec: {mean_prec[i]:.4f}")
        
    print("-" * 80)
    print(f"MEAN (AVG) | Dice: {np.nanmean(mean_dice):.4f} | HD95: {np.nanmean(mean_hd95):7.2f} | IoU: {np.nanmean(mean_iou):.4f} | Sens: {np.nanmean(mean_sens):.4f} | Prec: {np.nanmean(mean_prec):.4f}")
    
    print("\n" + "="*80)
    print("🌟 CHỈ SỐ GỘP VÙNG (CHUẨN MICCAI BRATS 2024 GLI)")
    print("="*80)
    print(f"🔥 WT (Whole Tumor - 1+2+3)   | Dice: {np.nanmean(all_wt_dice):.4f} | HD95: {np.nanmean(all_wt_hd95):.2f}")
    print(f"🔥 TC (Tumor Core  - 1+3)     | Dice: {np.nanmean(all_tc_dice):.4f} | HD95: {np.nanmean(all_tc_hd95):.2f}")
    print(f"🔥 ET (Enhancing   - Nhãn 3)    | Dice: {np.nanmean(all_et_dice):.4f} | HD95: {np.nanmean(all_et_hd95):.2f}")
    print("="*80)
    
    return mean_dice, mean_hd95

def remove_small_connected_components(volume):
    '''
    Bộ lọc Thể tích Nâng cao (Dynamic Thresholding): 
    Cài đặt màng lọc RIÊNG BIỆT cho từng loại u để tránh xóa nhầm u nhỏ.
    '''
    # Ngưỡng kích thước (pixels) cho từng Class:
    min_sizes = {
        1: 20,   # NETC (Đỏ): Giữ lại các đốm >= 20
        2: 400,  # SNFH (Lục): Phù nề rất to, xóa mạnh tay các đốm rác < 400
        3: 10,   # ET (Vàng): Lõi ác tính cực kỳ bé, chỉ xóa rác < 10
        4: 20    # RC (Lam): Hốc mổ, giữ lại >= 20
    }
    
    cleaned_volume = np.zeros_like(volume)
    for c in np.unique(volume):
        if c == 0:
            continue
            
        threshold = min_sizes.get(c, 50) # Mặc định 50
        
        class_mask = (volume == c)
        labeled_mask, num_features = label(class_mask)
        for i in range(1, num_features + 1):
            component = (labeled_mask == i)
            if np.sum(component) >= threshold:
                cleaned_volume[component] = c
    return cleaned_volume

In [ ]:
# Run Evaluation on Validation/Test Set
# Un-comment to load best model and evaluate:

# checkpoint = torch.load(CONFIG['best_model_path'], map_location=device)
# model = UNet2p5D(CONFIG['in_channels'], CONFIG['num_classes'], CONFIG['base_channels']).to(device)
# model.load_state_dict(checkpoint['model_state'])
# print(f"Loaded checkpoint from epoch {checkpoint['epoch']} with validation Dice {checkpoint['val_dice']:.4f}")

# _, _, _, val_patients = build_dataloaders(CONFIG)
# evaluate_test_set(model, val_patients, CONFIG, device)
